# Loan Data Processing

## Objective
This notebook preprocesses the loan dataset by:

- Loading and inspecting data
- Cleaning currency formatting
- Fixing category errors
- Handling missing values
- Removing duplicates
- Handling outliers
- Label encoding
- Feature engineering
- Feature scaling
- Saving cleaned dataset

In [ ]:
# =====================================
# Assignment Five Part B1
# Loan Data Processing
# =====================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

print("Libraries imported")


# Load Dataset

df = pd.read_csv("raw_loan_dataset.csv")

print("\nFirst Rows")
print(df.head())

print("\nInfo")
print(df.info())

print("\nMissing Values")
print(df.isnull().sum())


# Clean Currency Columns

for col in ["Income","LoanAmount"]:

    df[col]=(
        df[col]
        .astype(str)
        .str.replace("$","",regex=False)
        .str.replace(",","",regex=False)
    )

    df[col]=pd.to_numeric(
        df[col],
        errors="coerce"
    )


print("\nCurrency cleaned")


# Fix Category Errors

yes_values=["yes","y","approved"]
no_values=["no","n","rejected"]

for col in [
"HasCollateral",
"PreviousDefaults",
"Approved"
]:

    df[col]=(
        df[col]
        .astype(str)
        .str.lower()
        .str.strip()
    )


    df[col]=df[col].replace(
        yes_values,
        "Yes"
    )


    df[col]=df[col].replace(
        no_values,
        "No"
    )


    print(f"\n{col}")

    print(
    df[col].value_counts()
    )


# Missing Values

numeric=[

"Income",
"CreditScore",
"LoanAmount",
"EmploymentYears"

]

categorical=[

"HasCollateral",
"PreviousDefaults",
"Approved"

]


for col in numeric:

    df[col]=df[col].fillna(
    df[col].median()
    )


for col in categorical:

    df[col]=df[col].fillna(
    df[col].mode()[0]
    )


print("\nAfter Missing")

print(df.isnull().sum())


# Remove Duplicates

before=df.shape[0]

df=df.drop_duplicates()

after=df.shape[0]

print("\nBefore:",before)

print("After:",after)


# IQR Outliers

cols=[

"Income",
"CreditScore",
"LoanAmount",
"EmploymentYears"

]


for col in cols:

    q1=df[col].quantile(.25)

    q3=df[col].quantile(.75)

    iqr=q3-q1

    lower=q1-(1.5*iqr)

    upper=q3+(1.5*iqr)

    df[col]=df[col].clip(
    lower,
    upper
    )


print("\nOutliers capped")


# Label Encoding

mapping={

"Yes":1,
"No":0

}


for col in [

"Approved",
"HasCollateral",
"PreviousDefaults"

]:

    df[col]=df[col].map(
    mapping
    )


print("\nApproved Distribution")

print(
df["Approved"].value_counts()
)


print(
df["Approved"].value_counts(
normalize=True
)
)


# Feature Engineering

df["DebtToIncome"]=(
df["LoanAmount"]
/
df["Income"]
)


df["IncomePerYearEmployed"]=(
df["Income"]
/
(df["EmploymentYears"]+1)
)


print("\nFeatures Added")


# Scaling

"""
Chosen Scaler:
RobustScaler

Reason:
Uses median and IQR.
Less sensitive to outliers.

Source:
Scikit-learn documentation
"""

scaler=RobustScaler()

scale_cols=[

"Income",
"CreditScore",
"LoanAmount",
"EmploymentYears",
"DebtToIncome",
"IncomePerYearEmployed"

]


df[scale_cols]=scaler.fit_transform(
df[scale_cols]
)


print("\nScaling completed")


print(df.info())

print(df.isnull().sum())

print(df.head())


df.to_csv(
"clean_loan_dataset.csv",
index=False
)

print(
"\nSaved clean_loan_dataset.csv"
)